# Estratégias de coleta

Toda rota começa na ENTRADA e termina no CAIXA. Cada estratégia recebe os slots do pedido, a matriz de distâncias e a loja de `grafo.ipynb`, e devolve a ordem de visita.

| Estratégia | Ordem de visita |
|---|---|
| `ordem_carrinho` | ordem em que o cliente colocou as seções no carrinho (referência) |
| `corredores` | varredura em S: corredores em ordem crescente, alternando o sentido |
| `vizinho` | sempre a seção mais próxima ainda não visitada |
| `custoMin` | rota de custo mínimo: Held-Karp até 15 seções e, acima disso, 2-opt |

In [1]:
ESTRATEGIAS = ("ordem_carrinho", "corredores", "vizinho", "custoMin")
LIMITE_HELD_KARP = 15


def custo_rota(ordem, distancias, loja):
    rota = [loja.ENTRADA, *ordem, loja.CAIXA]
    return sum(distancias[a, b] for a, b in zip(rota, rota[1:]))


def ordem_do_carrinho(slots, distancias, loja):
    return list(slots)


def varredura_por_corredores(slots, distancias, loja):
    por_corredor = {}
    for slot in slots:
        por_corredor.setdefault(loja.corredor[slot], []).append(slot)

    ordem = []
    for i, corredor in enumerate(sorted(por_corredor)):
        do_corredor = sorted(por_corredor[corredor], key=lambda slot: loja.posicao[slot])
        if i % 2 == 1:
            do_corredor.reverse()
        ordem += do_corredor
    return ordem


def vizinho_mais_proximo(slots, distancias, loja):
    faltam = sorted(slots)
    atual = loja.ENTRADA
    ordem = []
    while faltam:
        atual = min(faltam, key=lambda slot: (distancias[atual, slot], slot))
        ordem.append(atual)
        faltam.remove(atual)
    return ordem

## Rota de custo mínimo

Achar a ordem de menor custo é um caixeiro viajante em caminho, com início na ENTRADA e fim no CAIXA.

- **Held-Karp:** programação dinâmica sobre subconjuntos. `custo[S][j]` guarda o menor custo para sair da ENTRADA, visitar o conjunto `S` e parar em `j`; a rota ótima é reconstruída de trás para frente por `anterior`. O custo cresce como $O(k^2 2^k)$, por isso o limite de 15 seções.
- **2-opt:** inverte trechos da rota enquanto isso reduzir a distância. Parte de uma rota pronta e nunca piora, então o resultado fica no máximo igual ao ponto de partida.

Na rota_custo_minimo, acima de 15 seções o Held-Karp fica caro demais, então o 2-opt melhora as rotas do vizinho e da varredura, e a menor das duas é a escolhida.

In [ ]:
def dois_opt(ordem, distancias, loja):
    rota = [loja.ENTRADA, *ordem, loja.CAIXA]
    melhorou = True
    while melhorou:
        melhorou = False
        for i in range(1, len(rota) - 2):
            for j in range(i + 1, len(rota) - 1):
                antes = distancias[rota[i - 1], rota[i]] + distancias[rota[j], rota[j + 1]]
                depois = distancias[rota[i - 1], rota[j]] + distancias[rota[i], rota[j + 1]]
                if depois < antes - 1e-9:
                    rota[i:j + 1] = reversed(rota[i:j + 1])
                    melhorou = True
    return rota[1:-1]


def held_karp(slots, distancias, loja):
    n = len(slots)
    custo = [[float("inf")] * n for _ in range(1 << n)]
    anterior = [[-1] * n for _ in range(1 << n)]
    for j in range(n):
        custo[1 << j][j] = distancias[loja.ENTRADA, slots[j]]

    for visitados in range(1, 1 << n):
        for j in range(n):
            sem_j = visitados ^ (1 << j)
            if not visitados & (1 << j) or sem_j == 0:
                continue
            for i in range(n):
                if sem_j & (1 << i):
                    novo_custo = custo[sem_j][i] + distancias[slots[i], slots[j]]
                    if novo_custo < custo[visitados][j]:
                        custo[visitados][j] = novo_custo
                        anterior[visitados][j] = i

    todos = (1 << n) - 1
    j = min(range(n), key=lambda j: custo[todos][j] + distancias[slots[j], loja.CAIXA])
    ordem = []
    visitados = todos
    while j != -1:
        ordem.append(slots[j])
        visitados, j = visitados ^ (1 << j), anterior[visitados][j]
    return ordem[::-1]


def usa_held_karp(slots):
    return len(slots) <= LIMITE_HELD_KARP


def rota_custo_minimo(slots, distancias, loja):
    if usa_held_karp(slots):
        return held_karp(slots, distancias, loja)
    candidatas = [dois_opt(vizinho_mais_proximo(slots, distancias, loja), distancias, loja),
                  dois_opt(varredura_por_corredores(slots, distancias, loja), distancias, loja)]
    return min(candidatas, key=lambda ordem: custo_rota(ordem, distancias, loja))


FUNCOES = {
    "ordem_carrinho": ordem_do_carrinho,
    "corredores": varredura_por_corredores,
    "vizinho": vizinho_mais_proximo,
    "custoMin": rota_custo_minimo,
}


def executar(estrategia, slots, distancias, loja):
    ordem = FUNCOES[estrategia](list(dict.fromkeys(slots)), distancias, loja)
    return ordem, custo_rota(ordem, distancias, loja)